In [ ]:
!pip install healpy -q
print('healpy ready')


# v75 -- CMB Lensing m-Parity with Realistic LCDM Sims
**Author:** Andrew Cotting | 4 April 2026 | github.com/Andrew-Cot/KTF3-Analysis

Fixes D29 Caveat 1: flat power spectrum replaced by realistic lensing Cl.
Requires: COM_Lensing_4096_R3.00 folder already uploaded (from v74h).

In [ ]:
import numpy as np, healpy as hp, matplotlib.pyplot as plt, os, glob

# Find the kappa alm file
search = [
    'MV_dat_klm.fits',
    'COM_Lensing_4096_R3.00/MV/dat_klm.fits',
    '/content/MV_dat_klm.fits',
    '/content/COM_Lensing_4096_R3.00/MV/dat_klm.fits',
]
search += glob.glob('**/dat_klm.fits', recursive=True)
search += glob.glob('/content/**/*.fits', recursive=True)

filename = None
for f in search:
    if os.path.exists(f):
        size = os.path.getsize(f)
        print(f'Found: {f} ({size/1e6:.0f} MB)')
        if size > 1e6:
            filename = f
            break

if filename is None:
    print('ERROR: No lensing file found!')
    print('Make sure COM_Lensing_4096_R3.00 folder is uploaded to Colab.')
    print('The file needed: COM_Lensing_4096_R3.00/MV/dat_klm.fits')
    raise FileNotFoundError('Upload COM_Lensing_4096_R3.00 folder first')

print(f'Using: {filename}')
alm_kappa = hp.read_alm(filename)
print(f'lmax = {hp.Alm.getlmax(len(alm_kappa))}')
nside = 128
kappa_map = hp.alm2map(alm_kappa, nside=nside, verbose=False)
print(f'Map loaded: Nside={nside}')


In [ ]:
# Mask + rotate
npix = hp.nside2npix(nside)
b_gal = 90 - np.degrees(hp.pix2ang(nside, np.arange(npix))[0])
mask = (np.abs(b_gal) >= 20).astype(float)
print(f'f_sky = {mask.mean():.3f}')
r = hp.Rotator(rot=[277, 90-31], coord=['G','G'])
kappa_rot = r.rotate_map_pixel(kappa_map * mask)
lmax = 10
alm_data = hp.map2alm(kappa_rot, lmax=lmax)

def mparity(alm, lmax):
    res = {}
    for ell in range(2, lmax+1):
        Po, Pe = 0, 0
        for m in range(ell+1):
            p = np.abs(alm[hp.Alm.getidx(lmax,ell,m)])**2*(1 if m==0 else 2)
            if m%2: Po+=p
            else: Pe+=p
        res[ell] = (Po-Pe)/(Po+Pe) if (Po+Pe) else 0
    return res

mp_data = mparity(alm_data, lmax)
print('m-parity computed.')
print('ell  S_data')
for e in range(2, lmax+1): print(f'{e:>4}  {mp_data[e]:>8.4f}')


In [ ]:
# Null test: REALISTIC vs FLAT sims
print('Null test: 1000 sims each (realistic + flat)...')

nside_s = 64
mask_s = hp.ud_grade(mask, nside_s)

# Realistic lensing Cl (Planck 2018 approximate)
cl_real = np.zeros(lmax+1)
for L in range(2, lmax+1):
    cl_real[L] = 2.4e-7 / (L*(L+1))**2  # realistic shape

# Flat Cl (v74h comparison)
cl_flat = np.zeros(lmax+1)
for L in range(2, lmax+1):
    cl_flat[L] = 1/(L*(L+1))

sims_real = {e:[] for e in range(2,lmax+1)}
sims_flat = {e:[] for e in range(2,lmax+1)}

for i in range(1000):
    for cl, sims in [(cl_real, sims_real), (cl_flat, sims_flat)]:
        sm = hp.synfast(cl, nside=nside_s, lmax=lmax, verbose=False)*mask_s
        sa = hp.map2alm(r.rotate_map_pixel(sm), lmax=lmax)
        mp_s = mparity(sa, lmax)
        for e in range(2,lmax+1): sims[e].append(mp_s[e])

print()
print(f'ell  S_data  sigma_real  sigma_flat  change')
print('-'*52)
sigmas_real = {}; sigmas_flat = {}
for e in range(2, lmax+1):
    ar = np.array(sims_real[e]); af = np.array(sims_flat[e])
    sr = (mp_data[e]-ar.mean())/ar.std()
    sf = (mp_data[e]-af.mean())/af.std()
    sigmas_real[e]=sr; sigmas_flat[e]=sf
    ch = 'UP' if abs(sr)>abs(sf) else 'down'
    print(f'{e:>4} {mp_data[e]:>7.4f} {sr:>11.3f} {sf:>11.3f}  {ch}')

m_real = np.mean([abs(sigmas_real[e]) for e in range(2,6)])
m_flat = np.mean([abs(sigmas_flat[e]) for e in range(2,6)])
print(f'\nMean |sigma| ell=2-5 (realistic): {m_real:.3f}')
print(f'Mean |sigma| ell=2-5 (flat v74h): {m_flat:.3f}')
print(f'CMB primary mean (v40):            0.700')
print(f'Result: {"CONSISTENT" if m_real>0.70 else "NULL"} '
      f'({"IMPROVED" if m_real>m_flat else "unchanged"} vs v74h)')


In [ ]:
# Plot
fig, axes = plt.subplots(1, 2, figsize=(14,6))
fig.suptitle('v75 -- Lensing m-Parity: Realistic vs Flat LCDM Sims\n'
             'KTF3 axis: l=277, b=-31 | REAL DATA (Planck PR3)', fontsize=12, fontweight='bold')

ells = list(range(2, lmax+1))
ax1 = axes[0]
ax1.plot(ells, [sigmas_real[e] for e in ells], 'g-o', lw=2, ms=8,
         label=f'Realistic sims (mean={m_real:.3f})')
ax1.plot(ells, [sigmas_flat[e] for e in ells], 'r--s', lw=1.5, ms=6, alpha=0.7,
         label=f'Flat sims / v74h (mean={m_flat:.3f})')
ax1.axhline(0, color='black')
ax1.axhline(1, color='orange', linestyle='--', label='1 sigma')
ax1.axhline(-1, color='orange', linestyle='--')
ax1.axhline(0.70, color='blue', linestyle=':', lw=1.5, label='CMB v40 (0.70)')
ax1.grid(True, alpha=0.3); ax1.legend(fontsize=9)
ax1.set_xlabel('Multipole ell'); ax1.set_ylabel('Sigma vs LCDM')
ax1.set_title('Sigma: realistic vs flat (fixes D29 Caveat 1)')

ax2 = axes[1]
diff = [sigmas_real[e]-sigmas_flat[e] for e in ells]
ax2.bar(ells, diff, color=['#2ecc71' if d>0 else '#e74c3c' for d in diff],
        alpha=0.85, edgecolor='black')
ax2.axhline(0, color='black')
ax2.grid(True, alpha=0.3)
ax2.set_xlabel('Multipole ell')
ax2.set_ylabel('Delta sigma (realistic - flat)')
ax2.set_title('Improvement from realistic sims\n(positive = signal stronger than v74h)')

plt.tight_layout()
plt.savefig('v75_lensing_realistic.png', dpi=150, bbox_inches='tight')
plt.show()
print('GitHub: github.com/Andrew-Cot/KTF3-Analysis | OSF: osf.io/42nks')
